# Minimal LoRA XLM-R Regressor + Classifier Ensemble Probe

Train a LoRA XLM-R regressor and a LoRA XLM-R classifier on the same train/validation split, then estimate whether an ensemble is worth pursuing by comparing their validation errors and an oracle that picks the better prediction per example.

This notebook is intentionally small: it answers whether the two models make complementary mistakes before adding any real ensemble machinery.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Sequence, Value
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoConfig, AutoTokenizer, Trainer, TrainingArguments, XLMRobertaModel, XLMRobertaPreTrainedModel, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from experiments.config import ModelConfig
from experiments.models import SentimentModel

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_regressor_classifier_ensemble_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Shared data split

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")
df["lang"] = df.get("lang", "unk")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
y_val = val_df["label"].to_numpy(dtype=int)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def one_hot(label, n_classes=N_CLASSES):
    vec = [0.0] * n_classes
    vec[int(label)] = 1.0
    return vec


def tokenize_regression(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [float(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out


def tokenize_classification(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [one_hot(x) for x in batch["label"]]
    langs = batch["lang"] if "lang" in batch else ["unk"] * len(batch["sentence"])
    out["lang"] = [0 if x == "eng_Latn" else 1 for x in langs]
    return out


def to_regression_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_regression, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("float32"))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds


def to_classification_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize_classification, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Sequence(Value("float32"), length=N_CLASSES))
    ds = ds.cast_column("lang", Value("int64"))
    ds.set_format("torch")
    return ds


reg_train_ds = to_regression_dataset(train_df)
reg_val_ds = to_regression_dataset(val_df)
cls_train_ds = to_classification_dataset(train_df)
cls_val_ds = to_classification_dataset(val_df)

## Shared helpers

In [ ]:
def make_training_args(run_name):
    kwargs = dict(
        output_dir=str(OUTPUT_DIR / run_name / "checkpoints"),
        overwrite_output_dir=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED,
    )
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)


def apply_thresholds(scores, thresholds):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    thresholds = np.asarray(thresholds, dtype=np.float32)
    return np.searchsorted(thresholds, scores, side="right").astype(int)


def tune_mae_thresholds(scores, labels, n_classes=N_CLASSES):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_labels = labels[order]

    unique_scores, group_starts = np.unique(sorted_scores, return_index=True)
    group_ends = np.r_[group_starts[1:], len(sorted_scores)]
    n_groups = len(unique_scores)

    group_cost = np.zeros((n_classes, n_groups), dtype=np.float64)
    for g, (start, end) in enumerate(zip(group_starts, group_ends)):
        y = sorted_labels[start:end]
        for cls in range(n_classes):
            group_cost[cls, g] = np.abs(cls - y).sum()

    prefix_cost = np.c_[np.zeros(n_classes), np.cumsum(group_cost, axis=1)]
    dp = np.full((n_classes, n_groups + 1), np.inf, dtype=np.float64)
    back = np.zeros((n_classes, n_groups + 1), dtype=int)
    dp[0] = prefix_cost[0]

    for cls in range(1, n_classes):
        best_value = np.inf
        best_split = 0
        for j in range(n_groups + 1):
            candidate = dp[cls - 1, j] - prefix_cost[cls, j]
            if candidate < best_value:
                best_value = candidate
                best_split = j
            dp[cls, j] = prefix_cost[cls, j] + best_value
            back[cls, j] = best_split

    cuts = []
    j = n_groups
    for cls in range(n_classes - 1, 0, -1):
        j = back[cls, j]
        cuts.append(j)
    cuts = cuts[::-1]

    thresholds = []
    eps = 1e-6
    for cut in cuts:
        if cut <= 0:
            thresholds.append(float(unique_scores[0] - eps))
        elif cut >= n_groups:
            thresholds.append(float(unique_scores[-1] + eps))
        else:
            thresholds.append(float((unique_scores[cut - 1] + unique_scores[cut]) / 2.0))

    tuned_preds = apply_thresholds(scores, thresholds)
    return np.array(thresholds, dtype=np.float32), tuned_preds


def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(cls - classes), axis=1) for cls in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)

## Train regressor

In [ ]:
def make_regressor():
    cfg = ModelConfig(
        kind="lora_bert",
        name=MODEL_ID,
        geometry="default",
        lora_r=128,
        lora_alpha=64,
        lora_dropout=0.01,
    )
    model = SentimentModel.from_config(cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"regressor trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def regression_metrics(eval_pred):
    logits, labels = eval_pred
    scores = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    rounded = np.rint(np.clip(scores, 0, 4))
    return {
        "mae": float(mean_absolute_error(labels, scores)),
        "rounded_mae": float(mean_absolute_error(labels, rounded)),
    }


reg_model = make_regressor()
reg_trainer = Trainer(
    model=reg_model,
    args=make_training_args("regressor_1epoch"),
    train_dataset=reg_train_ds,
    eval_dataset=reg_val_ds,
    compute_metrics=regression_metrics,
)
reg_trainer.train()

In [ ]:
reg_scores = reg_trainer.predict(reg_val_ds).predictions.reshape(-1)
reg_scores_clipped = np.clip(reg_scores, 0, 4)
reg_naive_labels = np.rint(reg_scores_clipped).astype(int)
reg_thresholds, reg_labels = tune_mae_thresholds(reg_scores_clipped, y_val)

print("reg raw MAE:", mean_absolute_error(y_val, reg_scores))
print("reg naive rounded MAE:", mean_absolute_error(y_val, reg_naive_labels))
print("reg tuned thresholds:", reg_thresholds.tolist())
print("reg tuned MAE:", mean_absolute_error(y_val, reg_labels))

## Train classifier

In [ ]:
class XLMRClassificationModel(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
        hidden = config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
            nn.Linear(hidden // 4, config.num_labels),
        )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            log_probs = F.log_softmax(logits, dim=-1)
            loss = -(labels.float() * log_probs).sum(dim=-1).mean()
        return {"loss": loss, "logits": logits}


def make_classifier():
    config = AutoConfig.from_pretrained(MODEL_ID, num_labels=N_CLASSES)
    model = XLMRClassificationModel.from_pretrained(MODEL_ID, config=config)
    lora_config = LoraConfig(
        r=128,
        lora_alpha=64,
        target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        modules_to_save=["classifier"],
        lora_dropout=0.01,
        task_type="SEQ_CLS",
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"classifier trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def classification_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    true = np.asarray(labels).argmax(axis=1)
    map_preds = probs.argmax(axis=1)
    bayes_preds = bayes_mae_decode(probs)
    return {
        "accuracy": float(accuracy_score(true, map_preds)),
        "map_mae": float(mean_absolute_error(true, map_preds)),
        "bayes_mae": float(mean_absolute_error(true, bayes_preds)),
    }


cls_model = make_classifier()
cls_trainer = Trainer(
    model=cls_model,
    args=make_training_args("classifier_1epoch"),
    train_dataset=cls_train_ds,
    eval_dataset=cls_val_ds,
    compute_metrics=classification_metrics,
)
cls_trainer.train()

In [ ]:
cls_logits = cls_trainer.predict(cls_val_ds).predictions
cls_probs = softmax_np(cls_logits)
cls_map_labels = cls_probs.argmax(axis=1).astype(int)
cls_labels = bayes_mae_decode(cls_probs)

print("cls MAP MAE:", mean_absolute_error(y_val, cls_map_labels))
print("cls Bayes-MAE labels MAE:", mean_absolute_error(y_val, cls_labels))

## Oracle ensemble diagnostic

This is not a deployable ensemble score. It estimates complementarity: how much could be gained if a future gating rule could choose the better model per example.

In [ ]:
reg_err = np.abs(reg_labels - y_val)
cls_err = np.abs(cls_labels - y_val)

oracle_err = np.minimum(reg_err, cls_err)

print("reg MAE:", reg_err.mean())
print("cls MAE:", cls_err.mean())
print("oracle MAE:", oracle_err.mean())
print("classifier better fraction:", np.mean(cls_err < reg_err))
print("same fraction:", np.mean(cls_err == reg_err))
print("classifier worse fraction:", np.mean(cls_err > reg_err))

In [ ]:
summary = pd.DataFrame(
    [
        {"model": "regressor_tuned", "mae": reg_err.mean(), "count_best_or_tied": int(np.sum(reg_err <= cls_err))},
        {"model": "classifier_bayes_mae", "mae": cls_err.mean(), "count_best_or_tied": int(np.sum(cls_err <= reg_err))},
        {"model": "oracle_min", "mae": oracle_err.mean(), "count_best_or_tied": len(y_val)},
    ]
)
display(summary)

pd.crosstab(
    pd.Series(reg_labels, name="reg_label"),
    pd.Series(cls_labels, name="cls_label"),
    margins=True,
)

## Is the classifier mostly helping near regression thresholds?

In [ ]:
nearest_threshold_distance = np.min(np.abs(reg_scores_clipped[:, None] - reg_thresholds[None, :]), axis=1)
probe = pd.DataFrame(
    {
        "y": y_val,
        "reg_score": reg_scores_clipped,
        "reg_label": reg_labels,
        "cls_label": cls_labels,
        "reg_err": reg_err,
        "cls_err": cls_err,
        "classifier_better": cls_err < reg_err,
        "classifier_worse": cls_err > reg_err,
        "nearest_threshold_distance": nearest_threshold_distance,
    }
)

probe["threshold_bin"] = pd.cut(
    probe["nearest_threshold_distance"],
    bins=[-np.inf, 0.05, 0.10, 0.20, 0.50, np.inf],
    labels=["<=0.05", "0.05-0.10", "0.10-0.20", "0.20-0.50", ">0.50"],
)

threshold_summary = probe.groupby("threshold_bin", observed=True).agg(
    n=("y", "size"),
    reg_mae=("reg_err", "mean"),
    cls_mae=("cls_err", "mean"),
    classifier_better_fraction=("classifier_better", "mean"),
    classifier_worse_fraction=("classifier_worse", "mean"),
)
display(threshold_summary)

display(probe.sort_values("nearest_threshold_distance").head(20))

## Interpretation guide

| Result | Meaning |
|---|---|
| Oracle MAE much below 0.3849 | ensemble has potential |
| Oracle MAE only slightly below 0.3849 | classifier adds little |
| Classifier better mostly near thresholds | use classifier as edge tie-breaker |
| Classifier better randomly | hard to exploit |

In [ ]:
baseline = 0.3849
print("reg delta vs baseline:", reg_err.mean() - baseline)
print("cls delta vs baseline:", cls_err.mean() - baseline)
print("oracle delta vs baseline:", oracle_err.mean() - baseline)

if oracle_err.mean() < baseline - 0.02:
    print("Interpretation: oracle is meaningfully below baseline; ensembling may be worth exploring.")
elif oracle_err.mean() < baseline - 0.005:
    print("Interpretation: oracle is only modestly below baseline; try a tiny threshold/tie-break rule before anything fancy.")
else:
    print("Interpretation: oracle adds little on this split; keep the simpler model.")

## Direct gate: choose regressor or classifier

Instead of stacking labels into a new score, train a gate for whether the regressor has lower absolute error than the classifier. The out-of-fold gate probabilities are the main sanity check; the in-sample gate is optimistic and only shows whether the feature signal exists at all.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict


def ensure_minmax01(x):
    x = np.asarray(x, dtype=np.float64)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# Recompute these here so this cell does not depend on the earlier confidence-vote cells.
reg_conf_gate = np.abs(reg_scores_clipped[:, None] - reg_thresholds[None, :]).min(axis=1)
reg_conf_gate = ensure_minmax01(reg_conf_gate)
cls_entropy_gate = -(cls_probs * np.log(cls_probs + 1e-12)).sum(axis=1)
cls_conf_gate = ensure_minmax01(1.0 - cls_entropy_gate / np.log(N_CLASSES))
cls_expected_gate = cls_probs @ np.arange(N_CLASSES)
cls_margin_gate = np.sort(cls_probs, axis=1)[:, -1] - np.sort(cls_probs, axis=1)[:, -2]

X_gate = np.column_stack(
    [
        reg_scores_clipped,
        reg_labels,
        cls_expected_gate,
        cls_labels,
        reg_conf_gate,
        cls_conf_gate,
        cls_margin_gate,
        np.abs(reg_labels - cls_labels),
        reg_scores_clipped - cls_expected_gate,
        cls_probs,
    ]
)

gate_feature_names = [
    "reg_score",
    "reg_label",
    "cls_expected_score",
    "cls_label",
    "reg_threshold_conf",
    "cls_entropy_conf",
    "cls_margin",
    "label_disagreement",
    "score_disagreement",
    *[f"cls_prob_{k}" for k in range(N_CLASSES)],
]

reg_abs_err = np.abs(reg_labels - y_val)
cls_abs_err = np.abs(cls_labels - y_val)
gate_target = (reg_abs_err < cls_abs_err).astype(int)

print("gate target positive fraction, reg strictly better:", gate_target.mean())
print("tie fraction:", np.mean(reg_abs_err == cls_abs_err))

if len(np.unique(gate_target)) < 2:
    print("Gate target has only one class on this split; no gate can be trained.")
else:
    gate = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    p_reg_better_oof = cross_val_predict(gate, X_gate, gate_target, cv=cv, method="predict_proba")[:, 1]

    best_gate = None
    for tau in np.linspace(0.05, 0.95, 181):
        use_reg = p_reg_better_oof > tau
        pred = np.where(use_reg, reg_labels, cls_labels)
        mae = np.mean(np.abs(pred - y_val))
        if best_gate is None or mae < best_gate[0]:
            best_gate = (mae, tau, use_reg.mean())

    gate.fit(X_gate, gate_target)
    p_reg_better_in_sample = gate.predict_proba(X_gate)[:, 1]
    best_gate_in_sample = None
    for tau in np.linspace(0.05, 0.95, 181):
        use_reg = p_reg_better_in_sample > tau
        pred = np.where(use_reg, reg_labels, cls_labels)
        mae = np.mean(np.abs(pred - y_val))
        if best_gate_in_sample is None or mae < best_gate_in_sample[0]:
            best_gate_in_sample = (mae, tau, use_reg.mean())

    print("reg MAE:", reg_abs_err.mean())
    print("cls MAE:", cls_abs_err.mean())
    print("oracle MAE:", np.minimum(reg_abs_err, cls_abs_err).mean())
    print("direct gate OOF best MAE, tau, use_reg_fraction:", best_gate)
    print("direct gate in-sample best MAE, tau, use_reg_fraction:", best_gate_in_sample)

    display(
        pd.DataFrame({"feature": gate_feature_names, "coef": gate.coef_.ravel()})
        .sort_values("coef", key=np.abs, ascending=False)
    )

In [ ]:
if len(np.unique(gate_target)) >= 2:
    gate_tau = best_gate[1]
    gate_use_reg = p_reg_better_oof > gate_tau
    gate_labels = np.where(gate_use_reg, reg_labels, cls_labels).astype(int)

    gate_probe = pd.DataFrame(
        {
            "y": y_val,
            "reg_label": reg_labels,
            "cls_label": cls_labels,
            "gate_label": gate_labels,
            "p_reg_better_oof": p_reg_better_oof,
            "use_reg": gate_use_reg,
            "reg_err": reg_abs_err,
            "cls_err": cls_abs_err,
            "gate_err": np.abs(gate_labels - y_val),
        }
    )

    display(
        pd.DataFrame(
            [
                {"decoder": "regressor", "mae": reg_abs_err.mean()},
                {"decoder": "classifier", "mae": cls_abs_err.mean()},
                {"decoder": "oracle", "mae": np.minimum(reg_abs_err, cls_abs_err).mean()},
                {"decoder": "direct_gate_oof", "mae": np.abs(gate_labels - y_val).mean()},
            ]
        )
    )

    display(
        gate_probe.groupby("use_reg").agg(
            n=("y", "size"),
            gate_mae=("gate_err", "mean"),
            reg_mae=("reg_err", "mean"),
            cls_mae=("cls_err", "mean"),
            mean_p_reg_better=("p_reg_better_oof", "mean"),
        )
    )

    display(gate_probe.sort_values("p_reg_better_oof").head(10))
    display(gate_probe.sort_values("p_reg_better_oof", ascending=False).head(10))

## Constrained residual correction from classifier labels

Use the classifier prediction as the base label, then train a 3-class correction for `delta in {-1, 0, +1}`. This can predict labels that neither base model selected, but keeps the correction small.

In [ ]:
delta_target = np.clip(y_val - cls_labels, -1, 1).astype(int)
print("delta distribution:", pd.Series(delta_target).value_counts().sort_index().to_dict())

X_delta = np.column_stack(
    [
        cls_probs,
        cls_labels,
        cls_expected_gate,
        cls_conf_gate,
        cls_margin_gate,
        reg_scores_clipped,
        reg_labels,
        reg_conf_gate,
        reg_scores_clipped - cls_expected_gate,
        np.abs(reg_labels - cls_labels),
    ]
)

delta_feature_names = [
    *[f"cls_prob_{k}" for k in range(N_CLASSES)],
    "cls_label",
    "cls_expected_score",
    "cls_entropy_conf",
    "cls_margin",
    "reg_score",
    "reg_label",
    "reg_threshold_conf",
    "score_disagreement",
    "label_disagreement",
]

if len(np.unique(delta_target)) < 2:
    print("Delta target has only one class on this split; no residual model can be trained.")
else:
    delta_model = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000, multi_class="auto")
    delta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    delta_oof = cross_val_predict(delta_model, X_delta, delta_target, cv=delta_cv)
    residual_oof_labels = np.clip(cls_labels + delta_oof, 0, 4).astype(int)

    delta_model.fit(X_delta, delta_target)
    delta_in_sample = delta_model.predict(X_delta)
    residual_in_sample_labels = np.clip(cls_labels + delta_in_sample, 0, 4).astype(int)

    print("reg MAE:", mean_absolute_error(y_val, reg_labels))
    print("cls MAE:", mean_absolute_error(y_val, cls_labels))
    print("residual correction OOF MAE:", mean_absolute_error(y_val, residual_oof_labels))
    print("residual correction in-sample MAE:", mean_absolute_error(y_val, residual_in_sample_labels))

    display(
        pd.DataFrame(
            [
                {"decoder": "regressor", "mae": mean_absolute_error(y_val, reg_labels)},
                {"decoder": "classifier", "mae": mean_absolute_error(y_val, cls_labels)},
                {"decoder": "direct_gate_oof", "mae": mean_absolute_error(y_val, gate_labels) if "gate_labels" in globals() else np.nan},
                {"decoder": "residual_correction_oof", "mae": mean_absolute_error(y_val, residual_oof_labels)},
                {"decoder": "residual_correction_in_sample", "mae": mean_absolute_error(y_val, residual_in_sample_labels)},
            ]
        )
    )

    pd.crosstab(
        pd.Series(y_val, name="label"),
        pd.Series(residual_oof_labels, name="residual_oof_label"),
        margins=True,
    )